# Phase 2: Profiling & Pattern Discovery

Digging into key fields to find patterns worth investigating in Phase 3.

**Scope**: National Defence excluded (structurally different procurement). Only rows with valid `reporting_period` format.

In [ ]:
import os
import duckdb
import polars as pl
from dotenv import load_dotenv

load_dotenv(override=True)
dataset_path = os.getenv("LOCAL_DATASET_PATH")
con = duckdb.connect(database=":memory:")

con.execute("""
    CREATE TEMP TABLE contracts AS
    SELECT * FROM read_csv_auto(
        '{0}', delim=',', header=true, strict_mode=false, all_varchar=true, parallel=false
    )
""".format(dataset_path))

# Reusable view: excludes Defence, casts types, filters to valid reporting periods
con.execute("""
    CREATE TEMP VIEW data AS
    SELECT *,
        TRY_CAST(contract_value AS DOUBLE) AS val,
        TRY_CAST(original_value AS DOUBLE) AS orig_val,
        TRY_CAST(amendment_value AS DOUBLE) AS amend_val,
        LEFT(reporting_period, 9) AS fiscal_year,
        RIGHT(reporting_period, 2) AS quarter,
        CASE 
            WHEN reporting_period < '2019-2020' THEN 'Pre-2019'
            WHEN reporting_period >= '2019-2020' AND reporting_period < '2022-2023' THEN '2019-2022'
            WHEN reporting_period >= '2022-2023' THEN 'Post-2022'
        END AS era
    FROM contracts
    WHERE owner_org_title NOT LIKE 'National Defence%'
      AND reporting_period LIKE '____-____-Q_'
""")

print("Rows in analysis view:", con.sql("SELECT COUNT(*) FROM data").fetchone()[0])

Ready. Rows in analysis view: 814070


## Spending distribution

In [ ]:
# Contract value distribution (contracts only)
con.sql("""
    SELECT 
        CASE 
            WHEN val < 0 THEN 'Negative'
            WHEN val = 0 THEN '$0'
            WHEN val < 25000 THEN '< $25K'
            WHEN val < 100000 THEN '$25K-$100K'
            WHEN val < 1000000 THEN '$100K-$1M'
            WHEN val < 10000000 THEN '$1M-$10M'
            WHEN val >= 10000000 THEN '$10M+'
        END AS bucket,
        COUNT(*) AS contracts,
        ROUND(COUNT(*)*100.0/SUM(COUNT(*)) OVER(), 1) AS pct_count,
        ROUND(SUM(val)/1e9, 2) AS value_B,
        ROUND(SUM(val)*100.0/SUM(SUM(val)) OVER(), 1) AS pct_value
    FROM data
    WHERE instrument_type = 'C' AND val IS NOT NULL
    GROUP BY bucket
    ORDER BY MIN(val)
""").pl()

In [9]:
# Visualize: contract count vs value share by size bucket
import plotly.graph_objects as go
from plotly.subplots import make_subplots

BLUE, RED, ORANGE, GREEN, GRAY = '#4878A8', '#D04040', '#E8A040', '#6BAF6B', '#888888'

val_dist = con.sql("""
    SELECT 
        CASE 
            WHEN val < 25000 THEN '< $25K'
            WHEN val < 100000 THEN '$25K-$100K'
            WHEN val < 1000000 THEN '$100K-$1M'
            WHEN val < 10000000 THEN '$1M-$10M'
            WHEN val >= 10000000 THEN '$10M+'
        END AS bucket,
        ROUND(COUNT(*)*100.0/SUM(COUNT(*)) OVER(), 1) AS pct_count,
        ROUND(SUM(val)*100.0/SUM(SUM(val)) OVER(), 1) AS pct_value
    FROM data WHERE instrument_type='C' AND val > 0
    GROUP BY bucket ORDER BY MIN(val)
""").pl()

buckets = val_dist["bucket"].to_list()

fig = make_subplots(rows=1, cols=2, subplot_titles=("% of contracts by count", "% of total value"))

fig.add_trace(go.Bar(
    x=buckets, y=val_dist["pct_count"].to_list(),
    marker_color=BLUE,
    text=[f'{v}%' for v in val_dist["pct_count"].to_list()],
    textposition='outside',
    cliponaxis=False,
    showlegend=False
), row=1, col=1)

fig.add_trace(go.Bar(
    x=buckets, y=val_dist["pct_value"].to_list(),
    marker_color=RED,
    text=[f'{v}%' for v in val_dist["pct_value"].to_list()],
    textposition='outside',
    cliponaxis=False,
    showlegend=False
), row=1, col=2)

fig.update_layout(
    title_text="Most contracts are small, but most money is in a few large ones",
    template="plotly_white",
    margin=dict(t=60),
    height=500
)
fig.update_xaxes(tickangle=30)
fig.update_yaxes(title_text="%", row=1, col=1)
fig.update_yaxes(title_text="%", row=1, col=2)
fig.show()

Most contracts are small but most dollars are in a handful of large ones. Descriptive, not actionable on its own.

## Competition rates by era

The `solicitation_procedure` field was only mandatory after 2019, so cross-era comparisons need care.

In [ ]:
# Solicitation procedure by era (contracts only)
con.sql("""
    SELECT era, solicitation_procedure,
        COUNT(*) AS cnt,
        ROUND(COUNT(*)*100.0/SUM(COUNT(*)) OVER(PARTITION BY era), 1) AS pct
    FROM data
    WHERE instrument_type = 'C' AND era IS NOT NULL
    GROUP BY era, solicitation_procedure
    ORDER BY era, cnt DESC
""").pl()

Non-competitive (TN) rose from 18.4% pre-2019 to 41.5% post-2022, but pre-2019 has 41.5% unknown because the field was not mandatory. The 2019-2022 vs post-2022 comparison (37.4% to 41.5%) is more honest -- a real but modest increase.

The `limited_tendering_reason` field should explain why contracts are non-competitive.

In [ ]:
# Limited tendering reasons for non-competitive contracts by era
con.sql("""
    SELECT era, limited_tendering_reason, COUNT(*) AS cnt,
        ROUND(COUNT(*)*100.0/SUM(COUNT(*)) OVER(PARTITION BY era), 1) AS pct
    FROM data
    WHERE instrument_type = 'C' AND solicitation_procedure = 'TN' AND era IS NOT NULL
    GROUP BY era, limited_tendering_reason
    ORDER BY era, cnt DESC
""").pl()

Pre-2019, code "85" (low dollar-value) was the top reason at 51%, but it was discontinued in 2022. Without a replacement, departments started using "00" (None). The spike in "no reason given" is a policy gap, not negligence.

The non-competitive rate is complex across eras. Better as supporting context than a standalone insight.

## Q4 spending pattern

Canada's fiscal year ends March 31. Q4 (Jan-Mar) is the last quarter before budgets reset.

In [ ]:
# Quarterly pattern -- count, total value, average value
con.sql("""
    SELECT quarter,
        COUNT(*) AS contracts,
        ROUND(SUM(val)/1e9, 2) AS total_B,
        ROUND(AVG(val), 0) AS avg_value
    FROM data
    WHERE instrument_type = 'C'
    GROUP BY quarter ORDER BY quarter
""").pl()

In [13]:
# Visualize quarterly pattern - the signal that led to Insight 1
q = con.sql("""
    SELECT quarter, COUNT(*) AS contracts, ROUND(AVG(val), 0) AS avg_value
    FROM data WHERE instrument_type='C'
    GROUP BY quarter ORDER BY quarter
""").pl()

colors = [BLUE, BLUE, BLUE, RED]

fig = make_subplots(rows=1, cols=2, subplot_titles=("Contract count", "Average value"))

fig.add_trace(go.Bar(
    x=q["quarter"].to_list(), y=q["contracts"].to_list(),
    marker_color=colors,
    text=[f'{v/1000:.0f}K' for v in q["contracts"].to_list()],
    textposition='outside',
    cliponaxis=False,
    showlegend=False
), row=1, col=1)

fig.add_trace(go.Bar(
    x=q["quarter"].to_list(), y=q["avg_value"].to_list(),
    marker_color=colors,
    text=[f'${v/1000:.0f}K' for v in q["avg_value"].to_list()],
    textposition='outside',
    cliponaxis=False,
    showlegend=False
), row=1, col=2)

fig.update_layout(
    title_text="Q4 stands out on both count and value",
    template="plotly_white",
    margin=dict(t=60),
    height=500
)
fig.show()

32% more new contracts in Q4 across all years. In post-2019 data (mandatory reporting), the volume surge is +20% but the value impact is stronger. Both confirm the pattern.

**Caveats**: `reporting_period` is when the contract was *reported*, not awarded. Only mandatory after 2019-01-01. But the pattern is too large and consistent to be explained by reporting lag alone. Worth a deep dive in Phase 3.

This raises another question: are departments also expanding existing contracts at year-end?

In [ ]:
# Amendment rate by era
con.sql("""
    SELECT era,
        COUNT(*) AS total_rows,
        SUM(CASE WHEN instrument_type='A' THEN 1 ELSE 0 END) AS amendments,
        ROUND(SUM(CASE WHEN instrument_type='A' THEN 1 ELSE 0 END)*100.0/COUNT(*), 1) AS amend_pct
    FROM data WHERE era IS NOT NULL
    GROUP BY era ORDER BY era
""").pl()

In [15]:
# Visualize amendment rate by era
amend_era = con.sql("""
    SELECT era,
        ROUND(SUM(CASE WHEN instrument_type='A' THEN 1 ELSE 0 END)*100.0/COUNT(*), 1) AS amend_pct
    FROM data WHERE era IS NOT NULL GROUP BY era ORDER BY era
""").pl()

fig = go.Figure(go.Bar(
    x=amend_era["era"].to_list(), y=amend_era["amend_pct"].to_list(),
    marker_color=[BLUE, ORANGE, RED],
    text=[f'{v}%' for v in amend_era["amend_pct"].to_list()],
    textposition='outside',
    cliponaxis=False
))
fig.add_hline(y=25, line_dash="dash", line_color="gray", opacity=0.3)
fig.update_layout(
    title="Amendment rate jumped from 16% to 25% after 2019",
    yaxis_title="Amendment rate (%)",
    template="plotly_white",
    margin=dict(t=60),
    height=450
)
fig.show()

Amendment rate jumped from 16% to 25% post-2019. 1 in 4 rows is now an amendment. But rate alone does not tell us if these are small adjustments or major scope changes.

In [ ]:
# Contracts that more than doubled via amendments
con.sql("""
    SELECT COUNT(*) AS total_amended,
        SUM(CASE WHEN growth_pct > 100 THEN 1 ELSE 0 END) AS doubled_plus,
        ROUND(SUM(CASE WHEN growth_pct > 100 THEN 1 ELSE 0 END)*100.0/COUNT(*), 1) AS doubled_pct,
        SUM(CASE WHEN growth_pct > 500 THEN 1 ELSE 0 END) AS grew_5x_plus
    FROM (
        SELECT procurement_id,
            (TRY_CAST(MAX(contract_value) AS DOUBLE) / 
             NULLIF(TRY_CAST(MIN(original_value) AS DOUBLE), 0) - 1) * 100 AS growth_pct
        FROM data
        WHERE procurement_id IS NOT NULL
        GROUP BY procurement_id
        HAVING COUNT(*) > 1 AND COUNT(DISTINCT instrument_type) > 1
    ) sub
    WHERE growth_pct IS NOT NULL
""").pl()

In [ ]:
# Amendment rate by commodity type (post-2019)
con.sql("""
    SELECT commodity_type,
        COUNT(*) AS total,
        SUM(CASE WHEN instrument_type='A' THEN 1 ELSE 0 END) AS amendments,
        ROUND(SUM(CASE WHEN instrument_type='A' THEN 1 ELSE 0 END)*100.0/COUNT(*), 1) AS amend_pct
    FROM data
    WHERE reporting_period >= '2019-2020' AND commodity_type IS NOT NULL
    GROUP BY commodity_type ORDER BY amend_pct DESC
""").pl()

In [ ]:
# Amendment rate: Q4 vs other quarters (post-2019)
con.sql("""
    SELECT 
        CASE WHEN quarter = 'Q4' THEN 'Q4' ELSE 'Q1-Q3' END AS period,
        ROUND(SUM(CASE WHEN instrument_type='A' THEN 1 ELSE 0 END)*100.0/COUNT(*), 1) AS amend_pct
    FROM data WHERE reporting_period >= '2019-2020'
    GROUP BY period ORDER BY period
""").pl()

Multiple signals pointing the same direction:
- Amendment rate jumped from 16% to 25%
- ~24% of amended contracts more than double in value
- Services and construction are amended 3x more than goods
- Amendments are more common in Q4 (28.2% vs 23.3%)

Year-end pressure drives both new awards and scope expansion on existing contracts. Worth a deep dive in Phase 3.

## Vendor concentration

In [ ]:
# Spending concentration: top 10, top 50, all others (post-2019 contracts)
con.sql("""
    WITH vendor_totals AS (
        SELECT vendor_name, 
            SUM(val) AS total_val,
            COUNT(*) AS contracts
        FROM data 
        WHERE instrument_type = 'C' AND reporting_period >= '2019-2020' AND vendor_name IS NOT NULL
        GROUP BY vendor_name
    ),
    ranked AS (
        SELECT *, ROW_NUMBER() OVER (ORDER BY total_val DESC) AS rn,
            SUM(total_val) OVER () AS grand_total
        FROM vendor_totals
    )
    SELECT 
        CASE WHEN rn <= 10 THEN 'Top 10' WHEN rn <= 50 THEN 'Top 11-50' ELSE 'All others' END AS vendor_group,
        COUNT(*) AS vendors,
        ROUND(SUM(total_val)/1e9, 2) AS value_B,
        ROUND(SUM(total_val)*100.0/MAX(grand_total), 1) AS pct_of_total
    FROM ranked
    GROUP BY CASE WHEN rn <= 10 THEN 'Top 10' WHEN rn <= 50 THEN 'Top 11-50' ELSE 'All others' END
    ORDER BY MIN(rn)
""").pl()

In [ ]:
# Amendment rate: top 50 vendors vs everyone else
con.sql("""
    WITH vendor_val AS (
        SELECT vendor_name, SUM(val) AS total_val
        FROM data WHERE instrument_type = 'C' AND reporting_period >= '2019-2020' AND vendor_name IS NOT NULL
        GROUP BY vendor_name
    ),
    top50 AS (
        SELECT vendor_name FROM vendor_val ORDER BY total_val DESC LIMIT 50
    )
    SELECT 
        CASE WHEN t.vendor_name IS NOT NULL THEN 'Top 50 vendors' ELSE 'All others' END AS vendor_group,
        COUNT(*) AS total_rows,
        SUM(CASE WHEN d.instrument_type = 'A' THEN 1 ELSE 0 END) AS amendments,
        ROUND(SUM(CASE WHEN d.instrument_type = 'A' THEN 1 ELSE 0 END)*100.0/COUNT(*), 1) AS amend_rate_pct
    FROM data d
    LEFT JOIN top50 t ON d.vendor_name = t.vendor_name
    WHERE d.reporting_period >= '2019-2020' AND d.vendor_name IS NOT NULL
    GROUP BY CASE WHEN t.vendor_name IS NOT NULL THEN 'Top 50 vendors' ELSE 'All others' END
    ORDER BY amend_rate_pct DESC
""").pl()

Top 50 vendors by value have a higher amendment rate (~27%) than others (~25%). Larger, longer-term contracts (services, construction) naturally get amended more. The vendors receiving the most money are the ones whose contracts change the most after award.

## Data quality issues

In [ ]:
# Malformed reporting_period values
con.sql("""
    SELECT reporting_period, COUNT(*) AS cnt
    FROM contracts
    WHERE reporting_period IS NOT NULL 
      AND reporting_period NOT LIKE '____-____-Q_'
    GROUP BY reporting_period ORDER BY cnt DESC
    LIMIT 10
""").pl()

In [ ]:
# Vendor name duplicates (same vendor, different spellings)
con.sql("""
    SELECT a.vendor_name AS name_1, b.vendor_name AS name_2,
           a.cnt + b.cnt AS combined_rows
    FROM (SELECT vendor_name, COUNT(*) AS cnt FROM contracts WHERE vendor_name IS NOT NULL GROUP BY vendor_name HAVING COUNT(*)>50) a
    JOIN (SELECT vendor_name, COUNT(*) AS cnt FROM contracts WHERE vendor_name IS NOT NULL GROUP BY vendor_name HAVING COUNT(*)>50) b
    ON UPPER(REPLACE(REPLACE(a.vendor_name,'.',''),' ','')) = UPPER(REPLACE(REPLACE(b.vendor_name,'.',''),' ',''))
    AND a.vendor_name < b.vendor_name
    ORDER BY combined_rows DESC
    LIMIT 5
""").pl()

**Issues found:**
1. **Malformed reporting_period** -- ~2,600 rows with values like "C", "Q1", "2108-2019". Filtered out with `LIKE '____-____-Q_'`.
2. **Vendor name inconsistency** -- same vendor, different spellings (case, periods). Affects vendor-level analysis.
3. **Era-dependent field availability** -- pre-2019 data is missing 35-97% on process fields. Not broken, just was not required. Eras analyzed separately.

## What I'm taking to Phase 3

| Selected | Why | Rejected patterns | Why not |
|---|---|---|---|
| **Q4 spending surge** | Clear signal, measurable, consistent | Non-competitive rate | Era effects make it hard to isolate |
| **Amendment growth** | Structural shift, extreme cases, ties into Q4 | Limited tendering "None" | Policy design issue, not actionable |
| **Vendor concentration** | Top vendors get amended more; connects to amendment insight | Vendor name duplicates | Data engineering fix, not analytical |